# Intergenerational GDM exerkine panel: RNA + protein sequence generation

Sequence-centric notebook for EXERKINEMAP. Steps:

1. **Panel**: cut a customized GDM panel out of the master exerkine manifest (tiers, axes, receptors).
2. **Resolve**: gene symbol to human RefSeq mRNA (NM_) and protein (NP_), MANE Select if available.
3. **Protein generation**: ESM-2 masked Gibbs variants around each natural protein.
4. **RNA generation**: back-translate each variant onto the natural CDS (native codons kept where residues are unchanged), native UTRs kept.
5. **Checks + export**: identity, CDS round-trip, FASTA files.

Generated sequences are *candidates*. Score them with the Sequence Generation Gym checks (ESMFold pLDDT, novelty, receptor binding, rat/human orthologs) before use.

In [23]:
!pip -q install biopython pandas numpy requests ViennaRNA torch transformers

In [2]:
import io, time, random, requests, numpy as np, pandas as pd
from collections import Counter, defaultdict
from itertools import groupby
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.Data import CodonTable
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Blast import NCBIWWW, NCBIXML
import RNA

Entrez.email = "you@example.com"   # <-- REQUIRED: set your email for NCBI

HUMAN, RAT = "Homo sapiens", "Rattus norvegicus"

# Customized for the Intergenerational GDM Exerkine Network Map
GENES = [
    # Core Exerkines / Myokines / Hepatokines
    "IL6", "BDNF", "IL15", "FGF21", "GDF15", "VEGFA", "FNDC5", "METRNL",
    "RARRES2", "LEP", "CFD", "FGF19", "FGF23", "RBP4", "SERPINA12", "NAMPT", 
    "RETN", "FABP4", "SPX", "IL10", "MSTN", "PPARGC1A", "TNF", "IL1B", "CCL2", 
    "ANGPTL4", "AHSG", "FETUB", "FGL1", "SHBG", "ENHO", "INHBE", "BGLAP", 
    "SOST", "CGA", "CGB", "CSH1", "CSH2", "PGF", "PAPPA", "RLN2", "IGF2", 
    "EGF", "HGF", "TGFB1", "IL33",
    # Receptor Panel
    "PGR", "ESR1", "ESR2", "GPER1", "RXFP1", "LEPR", "ADIPOR1", "ADIPOR2", 
    "IL6R", "IL6ST", "IL15RA", "IL2RB", "IL2RG", "NTRK2", "NPR1", "ITGAV", 
    "ITGB3", "ITGB5"
]

nt_aln = PairwiseAligner(); nt_aln.mode = "global"
nt_aln.match_score, nt_aln.mismatch_score = 1, -1
nt_aln.open_gap_score, nt_aln.extend_gap_score = -5, -1
aa_aln = PairwiseAligner(); aa_aln.mode = "global"
aa_aln.substitution_matrix = substitution_matrices.load("BLOSUM62")
aa_aln.open_gap_score, aa_aln.extend_gap_score = -10, -0.5

def identity(a, b, aligner):
    aln = aligner.align(a, b)[0]
    m = sum(a[i] == b[j] for (s1, e1), (s2, e2) in zip(*aln.aligned)
            for i, j in zip(range(s1, e1), range(s2, e2)))
    return m / min(len(a), len(b))

In [3]:
def ncbi_cds(gene, organism):
    term = f"{gene}[Gene Name] AND {organism}[Organism] AND srcdb_refseq_known[Properties] AND biomol_mrna[Properties]"
    ids = Entrez.read(Entrez.esearch(db="nuccore", term=term, retmax=1))["IdList"]
    time.sleep(0.4)
    if not ids: return None
    rec = SeqIO.read(Entrez.efetch(db="nuccore", id=ids[0], rettype="gb", retmode="text"), "genbank")
    time.sleep(0.4)
    for f in rec.features:
        if f.type == "CDS":
            return dict(acc=rec.id, gene=gene, cds=str(f.extract(rec.seq)).upper(),
                        protein=f.qualifiers.get("translation", [""])[0])
    return None

def uniprot_protein(gene, organism_id=9606):
    q = f"(gene_exact:{gene}) AND (organism_id:{organism_id}) AND (reviewed:true)"
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": q, "format": "fasta", "size": 1}, timeout=30)
    recs = list(SeqIO.parse(io.StringIO(r.text), "fasta"))
    return (recs[0].id.split("|")[1], str(recs[0].seq)) if recs else (None, None)

print(f"Fetching seeds for {len(GENES)} genes... this will take a moment.")
human = {g: ncbi_cds(g, HUMAN) for g in GENES}
rat   = {g: ncbi_cds(g, RAT) for g in GENES}
human = {g: v for g, v in human.items() if v}
print(f"Successfully fetched {len(human)} human RefSeq targets.")

tbl = CodonTable.unambiguous_dna_by_id[1]
syn = defaultdict(list)
for c, a in tbl.forward_table.items(): syn[a].append(c)
syn["*"] = list(tbl.stop_codons)

# codon usage is estimated from the seed panel itself
counts = Counter()
for v in human.values():
    s = v["cds"]; counts.update(s[i:i+3] for i in range(0, len(s) - 2, 3))
usage = {}
for aa, cods in syn.items():
    tot = sum(counts[c] + 1 for c in cods)
    usage[aa] = {c: (counts[c] + 1) / tot for c in cods}
w_rel = {aa: {c: f / max(d.values()) for c, f in d.items()} for aa, d in usage.items()}

FORBID = ["GGTCTC", "GAGACC", "CGTCTC", "GAGACG", "AATAAA", "TTTTT", "AAAAA", "GGGGG", "CCCCC"]
def gc(s): return (s.count("G") + s.count("C")) / len(s)
def longest_run(s): return max(len(list(g)) for _, g in groupby(s))
def has_forbidden(s): return any(m in s for m in FORBID)
def cai(s):
    ws = [w_rel[tbl.forward_table.get(s[i:i+3], "*")][s[i:i+3]] for i in range(0, len(s) - 2, 3)]
    return float(np.exp(np.mean(np.log(ws))))

def design(cds, T=1.5, gc_range=(0.45, 0.60), tries=300, seed=0):
    rng = random.Random(seed)
    prot = [tbl.forward_table.get(cds[i:i+3], "*") for i in range(0, len(cds) - 2, 3)]
    best = None
    for _ in range(tries):
        out = []
        for k, aa in enumerate(prot):
            if k == 0 or k == len(prot) - 1: out.append(cds[3*k:3*k+3]); continue  # keep start/stop
            cods = syn[aa]; p = np.array([usage[aa][c] ** (1 / T) for c in cods]); p /= p.sum()
            out.append(rng.choices(cods, weights=p)[0])
        s = "".join(out)
        if gc_range[0] <= gc(s) <= gc_range[1] and longest_run(s) < 6 and not has_forbidden(s):
            return s
        best = best or s
    return best

def fold_metrics(s, window=60):
    _, mfe = RNA.fold(s.replace("T", "U"))
    _, mfe5 = RNA.fold(s[:window].replace("T", "U"))
    return dict(mfe=mfe, mfe_5p=mfe5)

ATLAS_PATH = None   # Change if you want to cross-reference your ExerkineAtlas
ATLAS_GENE_COL = "gene"
atlas = pd.read_csv(ATLAS_PATH, sep=None, engine="python") if ATLAS_PATH else None
def atlas_row(gene):
    if atlas is None: return {"in_atlas": None}
    hit = atlas[atlas[ATLAS_GENE_COL].astype(str).str.upper() == gene.upper()]
    return {"in_atlas": len(hit) > 0, "atlas_rows": len(hit)}

Fetching seeds for 64 genes... this will take a moment.
Successfully fetched 64 human RefSeq targets.


In [5]:
print("Designing sequences...")
rows = []
for gene, seed in human.items():
    parent, prot = seed["cds"], seed["protein"]
    up_acc, up_prot = uniprot_protein(gene)
    rat_cds = (rat.get(gene) or {}).get("cds")
    
    # Running 10 design iterations per gene to save time on a 60-gene panel
    for i in range(10): 
        v = design(parent, T=1.5, seed=i)
        tr = str(Seq(v).translate(to_stop=True))
        row = dict(gene=gene, design=i, cds=v, len_nt=len(v),
                   translates_ok=(tr == prot.rstrip("*")),
                   uniprot_id=identity(tr, up_prot, aa_aln) if up_prot else np.nan,
                   id_human_cds=identity(v, parent, nt_aln),
                   id_rat_cds=identity(v, rat_cds, nt_aln) if rat_cds else np.nan,
                   gc=gc(v), cai=cai(v), run=longest_run(v), forbidden=has_forbidden(v),
                   **fold_metrics(v), **atlas_row(gene))
        rows.append(row)

df = pd.DataFrame(rows)
df["pass"] = df.translates_ok & ~df.forbidden & (df.run < 6) & df.gc.between(0.45, 0.60)

# Rank passing designs
ranked = (df[df["pass"]].sort_values(["gene", "cai", "mfe_5p"], ascending=[True, False, False])
            .groupby("gene").head(1).reset_index(drop=True))

# Optional: Run BLAST on top hits (Warning: This will take several minutes for ~60 genes)
def blastn_top(s):
    rec = NCBIXML.read(NCBIWWW.qblast("blastn", "refseq_rna", s, hitlist_size=3, megablast=False))
    if not rec.alignments: return dict(blast_top="no hit", blast_id=0.0, blast_cov=0.0)
    a = rec.alignments[0]; h = a.hsps[0]
    return dict(blast_top=a.hit_def[:70], blast_id=h.identities / h.align_length, blast_cov=h.align_length / len(s))

print("Running BLASTn verification on top designs...")
ranked = pd.concat([ranked, pd.DataFrame([blastn_top(s) for s in ranked.cds])], axis=1)

display(ranked.drop(columns="cds"))

Designing sequences...
Running BLASTn verification on top designs...


,gene,design,len_nt,translates_ok,uniprot_id,id_human_cds,id_rat_cds,gc,cai,run,forbidden,mfe,mfe_5p,in_atlas,pass,blast_top,blast_id,blast_cov
0,ADIPOR1,3,1128,True,1.000000,0.780142,0.763298,0.523050,0.718482,4,False,-384.299988,-13.400000,None,True,Hyperoodon ampullatus adiponectin receptor pro...,0.789007,1.000000
1,AHSG,5,1101,True,1.000000,0.760218,0.632672,0.553134,0.696305,4,False,-368.100006,-12.300000,None,True,"Homo sapiens alpha 2-HS glycoprotein (AHSG), t...",0.760870,1.002725
2,ANGPTL4,8,1107,True,0.997283,0.774164,0.686540,0.564589,0.724970,4,False,-429.600006,-16.500000,None,True,PREDICTED: Pan troglodytes angiopoietin like 4...,0.775862,0.995483
3,BDNF,1,789,True,1.000000,0.764259,0.768061,0.533587,0.763893,4,False,-266.799988,-10.000000,None,True,PREDICTED: Manis javanica brain derived neurot...,0.775862,0.955640
4,BGLAP,6,303,True,1.000000,0.788779,0.706667,0.590759,0.741685,4,False,-107.400002,-12.200000,None,True,PREDICTED: Nomascus leucogenys bone gamma-carb...,0.798680,1.000000
5,CCL2,1,300,True,1.000000,0.816667,0.620000,0.543333,0.797603,4,False,-85.300003,-6.700000,None,True,PREDICTED: Pan troglodytes C-C motif chemokine...,0.816667,1.000000
6,CFD,1,762,True,1.000000,0.755906,0.654856,0.599738,0.700690,4,False,-325.399994,-16.400000,None,True,"Homo sapiens complement factor D (CFD), transc...",0.755906,1.000000
7,CGA,1,444,True,1.000000,0.772523,0.685950,0.527027,0.771396,4,False,-121.400002,-4.400000,None,True,"Homo sapiens glycoprotein hormones, alpha poly...",0.772523,1.000000
8,CGB,7,498,True,1.000000,0.771084,NaN,0.584337,0.734226,4,False,-177.800003,-19.799999,None,True,Homo sapiens chorionic gonadotropin subunit be...,0.775551,1.002008
9,CSH1,6,654,True,1.000000,0.764526,0.507645,0.516820,0.737287,4,False,-198.199997,-17.600000,None,True,Homo sapiens chorionic somatomammotropin hormo...,0.764526,1.000000


In [4]:
def write_fasta(frame, path, rna=True):
    with open(path, "w") as f:
        for _, r in frame.iterrows():
            seq = r.cds.replace("T", "U") if rna else r.cds
            f.write(f">{r.gene}|design{r.design}|cai={r.cai:.2f}|gc={r.gc:.2f}|len={r.len_nt}\n{seq}\n")

passing = df[df["pass"]].sort_values(["gene", "cai"], ascending=[True, False])
write_fasta(ranked,  "gdm_exerkines_best_rna.fasta", rna=True)
write_fasta(ranked,  "gdm_exerkines_best_dna.fasta", rna=False)
write_fasta(passing, "gdm_exerkines_all_passing_rna.fasta", rna=True)

try:
    from google.colab import files
    for p in ["gdm_exerkines_best_rna.fasta", "gdm_exerkines_best_dna.fasta", "gdm_exerkines_all_passing_rna.fasta"]:
        files.download(p)
except ImportError:
    pass
    
print("Export complete.")

NameError: name 'df' is not defined

## 0. Setup

In [1]:
# Colab / fresh env only (comment out if your env already has these)
# !pip -q install biopython transformers torch pandas matplotlib
import csv, gzip, json, random, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

## 1. Configuration

In [20]:
MANIFEST   = "exerkines_manifest.csv"        # master manifest (the CSV with entity_id, gene_symbol, ...)
MANE_FILE  = None                           # optional: path to MANE.GRCh38.v*.summary.txt.gz (NCBI RefSeq/MANE)
EMAIL      = "you@example.edu"              # required by NCBI Entrez
OUT        = Path("gdm_panel_out"); OUT.mkdir(exist_ok=True)
CACHE      = OUT / "genbank_cache"; CACHE.mkdir(exist_ok=True)

ESM_MODEL  = "facebook/esm2_t12_35M_UR50D"  # swap for esm2_t33_650M_UR50D on a GPU
N_VARIANTS = 5
MASK_FRAC  = 0.10                           # fraction of positions resampled per variant (Met1 is never mutated)
GIBBS_STEPS = 2
TEMPERATURE = 1.0
SEED       = 0
TIERS      = [1, 2, 3]                      # which panel tiers to generate for
TEST_N     = None                           # set e.g. 5 to smoke-test on the first 5 genes

Entrez.email = EMAIL
rng = random.Random(SEED)

## 2. Customized GDM panel

Curated selection (edit `SEL` freely). Names, aliases and categories are pulled from the manifest.
`PROPOSED` rows are **not** in the manifest. `FGFR21` there is not a real symbol: FGF21 signals through FGFR1c + KLB.

In [21]:
# symbol: (axis, tier, role)
SEL = {
 # Tier 1 - manifest-tagged gdm_network
 "RARRES2":("maternal_adipokine",1,"ligand"), "LEP":("maternal_adipokine",1,"ligand"),
 "CFD":("maternal_adipokine",1,"ligand"), "FGF19":("maternal_hepatokine",1,"ligand"),
 "FGF21":("maternal_hepatokine",1,"ligand"), "FGF23":("maternal_osteokine",1,"ligand"),
 "FNDC5":("maternal_myokine",1,"ligand"),
 # Tier 2 - exercise-responsive / adipo- / hepato- / myokines relevant to glycemia & pregnancy
 "ADIPOQ":("maternal_adipokine",2,"ligand"), "RBP4":("maternal_adipokine",2,"ligand"),
 "ITLN1":("maternal_adipokine",2,"ligand"), "SERPINA12":("maternal_adipokine",2,"ligand"),
 "NAMPT":("maternal_adipokine",2,"ligand"), "RETN":("maternal_adipokine",2,"ligand"),
 "FABP4":("maternal_adipokine",2,"ligand"), "SPX":("maternal_adipokine",2,"ligand"),
 "APLN":("maternal_adipokine",2,"ligand"), "APELA":("placental_endocrine",2,"ligand"),
 "IL6":("maternal_myokine",2,"ligand"), "IL15":("maternal_myokine",2,"ligand"),
 "IL10":("maternal_myokine",2,"ligand"), "IGF1":("maternal_myokine",2,"ligand"),
 "GDF15":("maternal_myokine",2,"ligand"), "METRNL":("maternal_myokine",2,"ligand"),
 "BDNF":("maternal_myokine",2,"ligand"), "FSTL1":("maternal_myokine",2,"ligand"),
 "VEGFA":("maternal_myokine",2,"ligand"), "MSTN":("maternal_myokine",2,"ligand"),
 "FST":("maternal_myokine",2,"ligand"), "PPARGC1A":("maternal_myokine",2,"regulator"),
 "TNF":("inflammatory",2,"ligand"), "IL1B":("inflammatory",2,"ligand"),
 "CCL2":("inflammatory",2,"ligand"),
 "ANGPTL4":("maternal_hepatokine",2,"ligand"), "AHSG":("maternal_hepatokine",2,"ligand"),
 "FETUB":("maternal_hepatokine",2,"ligand"), "FGL1":("maternal_hepatokine",2,"ligand"),
 "SHBG":("maternal_hepatokine",2,"ligand"), "SELENOP":("maternal_hepatokine",2,"ligand"),
 "ENHO":("maternal_hepatokine",2,"ligand"), "INHBE":("maternal_hepatokine",2,"ligand"),
 "BGLAP":("maternal_osteokine",2,"ligand"), "SOST":("maternal_osteokine",2,"ligand"),
 # Tier 3 - placental / maternal-fetal interface
 "CGA":("placental_endocrine",3,"ligand"), "CGB":("placental_endocrine",3,"ligand"),
 "CSH1":("placental_endocrine",3,"ligand"), "CSH2":("placental_endocrine",3,"ligand"),
 "PGF":("placental_endocrine",3,"ligand"), "PAPPA":("placental_endocrine",3,"protease"),
 "RLN2":("placental_endocrine",3,"ligand"), "IGF2":("fetal_growth",3,"ligand"),
 "EGF":("fetal_growth",3,"ligand"), "HGF":("fetal_growth",3,"ligand"),
 "TGFB1":("placental_endocrine",3,"ligand"), "IL33":("inflammatory",3,"ligand"),
 "PGR":("placental_endocrine",3,"receptor"), "ESR1":("placental_endocrine",3,"receptor"),
 "ESR2":("placental_endocrine",3,"receptor"), "GPER1":("placental_endocrine",3,"receptor"),
 "RXFP1":("placental_endocrine",3,"receptor"),
 # Tier 3 - receptors for the Tier 1-2 ligands (needed for ligand-receptor edges)
 "LEPR":("receptor",3,"receptor"), "ADIPOR1":("receptor",3,"receptor"),
 "ADIPOR2":("receptor",3,"receptor"), "IL6R":("receptor",3,"receptor"),
 "IL6ST":("receptor",3,"receptor"), "IL15RA":("receptor",3,"receptor"),
 "IL2RB":("receptor",3,"receptor"), "IL2RG":("receptor",3,"receptor"),
 "NTRK2":("receptor",3,"receptor"), "NPR1":("receptor",3,"receptor"),
 "ITGAV":("receptor",3,"receptor"), "ITGB3":("receptor",3,"receptor"),
 "ITGB5":("receptor",3,"receptor"),
}
# Not in the master manifest. FGFR21 there is invalid: FGF21 signals via FGFR1c + KLB.
PROPOSED = [
 ("FGFR1","Fibroblast growth factor receptor 1","FGFR1","receptor","receptor",3,"FGF21/FGF19 co-receptor (replaces invalid FGFR21)"),
 ("KLB","Beta-klotho","KLB","receptor","receptor",3,"Obligate FGF21/FGF19 co-receptor"),
 ("INSR","Insulin receptor","INSR","receptor","receptor",3,"Core GDM glycemic axis"),
 ("INS","Insulin","INS","ligand","maternal_endocrine",3,"Core GDM glycemic axis"),
]

In [22]:
master = pd.read_csv(MANIFEST).fillna("")
by_sym = master[master.gene != ""].drop_duplicates("gene").set_index("gene")
missing = [s for s in SEL if s not in by_sym.index]
assert not missing, f"Not in manifest (or no gene): {missing}"

rows = []
for sym, (axis, tier, role) in SEL.items():
    r = by_sym.loc[sym]
    rows.append(dict(entity_id=r.entity_id, display_name=r.display_name, gene=sym, role=role, axis=axis,
                     tier=tier, aliases=r.aliases, manifest_categories=r.categories, source="manifest", notes=r.notes))
for eid, name, sym, role, axis, tier, note in PROPOSED:
    rows.append(dict(entity_id=eid, display_name=name, gene=sym, role=role, axis=axis, tier=tier,
                     aliases="", manifest_categories="", source="proposed_addition", notes=note))
panel = (pd.DataFrame(rows).sort_values(["tier", "axis", "gene"]).reset_index(drop=True))
panel.to_csv(OUT / "gdm_intergenerational_panel.csv", index=False)
print(len(panel), "entities")
display(panel.groupby(["tier", "axis"]).size().unstack(fill_value=0))
panel.head(10)

AssertionError: Not in manifest (or no gene): ['RARRES2', 'LEP', 'CFD', 'FGF19', 'FGF21', 'FGF23', 'RBP4', 'SERPINA12', 'NAMPT', 'RETN', 'FABP4', 'SPX', 'IL15', 'IL10', 'METRNL', 'MSTN', 'PPARGC1A', 'TNF', 'IL1B', 'CCL2', 'ANGPTL4', 'AHSG', 'FETUB', 'FGL1', 'SHBG', 'ENHO', 'INHBE', 'BGLAP', 'SOST', 'CGA', 'CGB', 'CSH1', 'CSH2', 'PGF', 'PAPPA', 'RLN2', 'IGF2', 'EGF', 'HGF', 'TGFB1', 'IL33', 'PGR', 'ESR1', 'ESR2', 'GPER1', 'RXFP1', 'LEPR', 'ADIPOR1', 'ADIPOR2', 'IL6R', 'IL6ST', 'IL15RA', 'IL2RB', 'IL2RG', 'NTRK2', 'NPR1', 'ITGAV', 'ITGB3', 'ITGB5']

In [14]:
print(master.columns)

Index(['name', 'tier', 'gene', 'status', 'refseq', 'uniprot', 'cds_nt',
       'protein_aa', 'translates_to_ncbi_protein', 'translates_to_uniprot',
       'note'],
      dtype='str')


## 3. Resolve RefSeq mRNA / protein

In [ ]:
def load_mane(path):
    out = {}
    if not path: return out
    with (gzip.open(path, "rt") if str(path).endswith(".gz") else open(path)) as f:
        hdr = f.readline().lstrip("#").rstrip("\n").split("\t"); ix = {k: i for i, k in enumerate(hdr)}
        for line in f:
            p = line.rstrip("\n").split("\t")
            if p[ix["MANE_status"]] == "MANE Select":
                out[p[ix["symbol"]]] = p[ix["RefSeq_nuc"]].split(".")[0]
    return out

MANE = load_mane(MANE_FILE)

def fetch_gb(acc_or_id):
    f = CACHE / f"{acc_or_id}.gb"
    if not f.exists():
        h = Entrez.efetch(db="nuccore", id=acc_or_id, rettype="gb", retmode="text")
        f.write_text(h.read()); h.close(); time.sleep(0.34)
    return SeqIO.read(f, "genbank")

def cds_feature(rec):
    return next((ft for ft in rec.features if ft.type == "CDS"), None)

def resolve(symbol):
    if symbol in MANE:
        return fetch_gb(MANE[symbol]), "MANE_Select"
    term = (f'{symbol}[Gene Name] AND "Homo sapiens"[Organism] AND refseq[filter] '
            f'AND biomol_mrna[prop] AND srcdb_refseq_known[prop]')
    h = Entrez.esearch(db="nuccore", term=term, retmax=20); ids = Entrez.read(h)["IdList"]; h.close(); time.sleep(0.34)
    best, best_len = None, -1
    for i in ids:
        rec = fetch_gb(i)
        if not rec.id.startswith("NM_"): continue
        genes = [q for ft in rec.features if ft.type == "gene" for q in ft.qualifiers.get("gene", [])]
        ft = cds_feature(rec)
        if symbol in genes and ft and len(ft) > best_len: best, best_len = rec, len(ft)
    return best, "longest_CDS_fallback"

def parse(rec):
    ft = cds_feature(rec); s, e = int(ft.location.start), int(ft.location.end)
    return dict(acc=rec.id, np=ft.qualifiers.get("protein_id", [""])[0],
                utr5=str(rec.seq[:s]).upper(), cds=str(rec.seq[s:e]).upper(), utr3=str(rec.seq[e:]).upper(),
                prot=ft.qualifiers["translation"][0])

In [ ]:
todo = panel[panel.tier.isin(TIERS)]
if TEST_N: todo = todo.head(TEST_N)

natural, log = {}, []
for _, r in todo.iterrows():
    try:
        rec, how = resolve(r.gene_symbol)
        if rec is None: log.append((r.entity_id, "failed", "no NM_ record")); continue
        d = parse(rec); d.update(how=how, row=r.to_dict()); natural[r.entity_id] = d
        assert str(Seq(d["cds"]).translate()).rstrip("*") == d["prot"], "CDS != annotated protein"
        log.append((r.entity_id, "resolved", f"{d['acc']} {d['np']} {how} aa={len(d['prot'])}"))
    except Exception as ex:
        log.append((r.entity_id, "failed", repr(ex)))

log_df = pd.DataFrame(log, columns=["entity_id", "status", "detail"]); log_df.to_csv(OUT / "resolution_log.csv", index=False)
print(log_df.status.value_counts().to_dict())
log_df[log_df.status != "resolved"]

In [ ]:
def hdr(eid, d):
    r = d["row"]; return f"{eid}|{r['gene_symbol']}|{d['acc']}|{d['np']}|tier{r['tier']}|{r['axis']}"

SeqIO.write([SeqRecord(Seq((d["utr5"] + d["cds"] + d["utr3"]).replace("T", "U")), id=hdr(e, d) + "|natural", description=d["how"])
             for e, d in natural.items()], OUT / "natural_mrna.fasta", "fasta")
SeqIO.write([SeqRecord(Seq(d["prot"]), id=hdr(e, d) + "|natural", description=d["how"])
             for e, d in natural.items()], OUT / "natural_protein.fasta", "fasta")
print("natural FASTA written:", len(natural))

## 4. Protein generation (ESM-2 masked Gibbs)

For each gene: pick `MASK_FRAC` of positions (never Met1), then repeatedly re-mask one position and resample it from the ESM-2 conditional distribution (special and ambiguous tokens excluded). Proteins longer than the 1022-residue ESM-2 window keep their tail unmodified.

In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(ESM_MODEL); esm = EsmForMaskedLM.from_pretrained(ESM_MODEL).to(device).eval()
BAD = set(tok.all_special_ids) | {tok.convert_tokens_to_ids(a) for a in "BJOUXZ"}
print("device:", device)

def gibbs_variant(prot, mask_frac=MASK_FRAC, steps=GIBBS_STEPS, temp=TEMPERATURE, max_len=1022):
    L = min(len(prot), max_len); seq = list(prot[:L])
    pos = rng.sample(range(1, L), min(max(1, int(mask_frac * L)), L - 1))
    for _ in range(steps):
        rng.shuffle(pos)
        for p in pos:
            inp = seq.copy(); inp[p] = tok.mask_token
            ids = tok("".join(inp), return_tensors="pt").to(device)
            with torch.no_grad(): logits = esm(**ids).logits[0, p + 1] / temp
            probs = torch.softmax(logits, -1)
            for t in BAD: probs[t] = 0
            seq[p] = tok.convert_ids_to_tokens(int(torch.multinomial(probs / probs.sum(), 1)))
    return "".join(seq) + prot[L:]

In [ ]:
variants = {}   # entity_id -> list of protein strings
for e, d in natural.items():
    variants[e] = [gibbs_variant(d["prot"]) for _ in range(N_VARIANTS)]
    print(d["row"]["gene_symbol"], end=" ", flush=True)

## 5. RNA generation (codon-aware back-translation)

Each variant is mapped back onto the natural CDS: unchanged residues keep their **native codon**, substituted residues sample human codon usage (Kazusa frequencies). The native 5'/3' UTRs and stop codon are retained, so differences from the natural mRNA are confined to the coding changes. A round-trip assertion confirms the CDS translates back to the generated protein.

In [ ]:
# Human codon usage (per-thousand, Kazusa); used only for substituted residues.
CODON_FREQ = {
 'A':{'GCC':27.7,'GCT':18.4,'GCA':15.8,'GCG':7.4},'R':{'AGA':12.2,'AGG':12.0,'CGG':11.4,'CGC':10.4,'CGA':6.2,'CGT':4.5},
 'N':{'AAC':19.1,'AAT':17.0},'D':{'GAC':25.1,'GAT':21.8},'C':{'TGC':12.6,'TGT':10.6},
 'Q':{'CAG':34.2,'CAA':12.3},'E':{'GAG':39.6,'GAA':29.0},'G':{'GGC':22.2,'GGA':16.5,'GGG':16.5,'GGT':10.8},
 'H':{'CAC':15.1,'CAT':10.9},'I':{'ATC':20.8,'ATT':16.0,'ATA':7.5},
 'L':{'CTG':39.6,'CTC':19.6,'TTG':12.9,'CTT':13.2,'TTA':7.7,'CTA':7.2},'K':{'AAG':31.9,'AAA':24.4},
 'M':{'ATG':22.0},'F':{'TTC':20.3,'TTT':17.6},'P':{'CCC':19.8,'CCT':17.5,'CCA':16.9,'CCG':6.9},
 'S':{'AGC':19.5,'TCC':17.7,'TCT':15.2,'AGT':12.1,'TCA':12.2,'TCG':4.4},'T':{'ACC':18.9,'ACA':15.1,'ACT':13.1,'ACG':6.1},
 'W':{'TGG':13.2},'Y':{'TAC':15.3,'TAT':12.2},'V':{'GTG':28.1,'GTC':14.5,'GTT':11.0,'GTA':7.1},
}

In [ ]:
def back_translate(nat_prot, var_prot, nat_cds):
    codons = [nat_cds[i:i+3] for i in range(0, len(nat_prot) * 3, 3)]
    out = [codons[i] if nat_prot[i] == var_prot[i] else rng.choices(list(CODON_FREQ[var_prot[i]]),
           weights=list(CODON_FREQ[var_prot[i]].values()))[0] for i in range(len(var_prot))]
    return "".join(out) + nat_cds[len(var_prot) * 3:]

def identity(a, b): return sum(x == y for x, y in zip(a, b)) / len(a)
def gc(s): return (s.count("G") + s.count("C")) / len(s)

gen_rows, gen_prot_recs, gen_rna_recs = [], [], []
for e, vs in variants.items():
    d = natural[e]
    for k, v in enumerate(vs, 1):
        cds = back_translate(d["prot"], v, d["cds"])
        assert str(Seq(cds).translate()).rstrip("*") == v, f"round-trip failed {e} gen{k}"
        vid = f"{hdr(e, d)}|gen{k}"; ident = identity(d["prot"], v)
        mrna = (d["utr5"] + cds + d["utr3"]).replace("T", "U")
        gen_prot_recs.append(SeqRecord(Seq(v), id=vid, description=f"aa_identity={ident:.3f}"))
        gen_rna_recs.append(SeqRecord(Seq(mrna), id=vid, description=f"aa_identity={ident:.3f}"))
        gen_rows.append(dict(entity_id=e, gene_symbol=d["row"]["gene_symbol"], tier=d["row"]["tier"], axis=d["row"]["axis"],
                             variant=k, aa_identity=ident, n_subs=sum(a != b for a, b in zip(d["prot"], v)),
                             cds_gc_natural=gc(d["cds"]), cds_gc_generated=gc(cds), mrna_len=len(mrna)))
gen_df = pd.DataFrame(gen_rows); gen_df.to_csv(OUT / "generated_summary.csv", index=False)
print(len(gen_df), "variants; round-trip OK")
gen_df.groupby("tier")[["aa_identity", "n_subs"]].describe().round(3)

## 6. Export and quick QC

In [ ]:
SeqIO.write(gen_prot_recs, OUT / "generated_protein.fasta", "fasta")
SeqIO.write(gen_rna_recs, OUT / "generated_mrna.fasta", "fasta")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for t, g in gen_df.groupby("tier"): ax[0].hist(g.aa_identity, bins=20, alpha=.6, label=f"tier {t}")
ax[0].set_xlabel("amino-acid identity to natural"); ax[0].legend()
ax[1].scatter(gen_df.cds_gc_natural, gen_df.cds_gc_generated, s=8); ax[1].plot([.3, .8], [.3, .8], "k--", lw=.7)
ax[1].set_xlabel("CDS GC natural"); ax[1].set_ylabel("CDS GC generated")
plt.tight_layout(); plt.savefig(OUT / "generation_qc.png", dpi=150); plt.show()
print(sorted(p.name for p in OUT.iterdir()))

## 7. Next: score the candidates

- ESMFold: pLDDT and RMSD to the natural structure (the Gym's structure check).
- Novelty: identity to the nearest natural sequence (NCBI / UniProt / ExerkineAtlas).
- Receptor binding: pair scorer on ESM-2 embeddings for ligand-receptor edges in this panel (e.g. LEP-LEPR, ADIPOQ-ADIPOR1/2, IL6-IL6R/IL6ST, FGF21-FGFR1/KLB).
- Rat vs human orthologs via UniProt.